# 03 — Il confronto

Tutti i modelli finiti (feature, baseline, transformer con i loro seed), sugli stessi fold **per gruppo**. Le
metriche principali, fissate prima dei risultati: **F1 macro a 5 classi** e **accuratezza binaria**, per
testata. Si riportano anche accuratezza a 5 classi, tolerant 0.5 e 1, e il dettaglio per livello.

Gli intervalli al 95% vengono da un **bootstrap per gruppo** (2000 ricampionamenti dei gruppi di testate: le
testate dello stesso gruppo sbagliano insieme). Per ogni transformer, la **differenza** con il classificatore a
feature (quello con C scelto in validazione interna, `feature_tuned`) sugli stessi ricampionamenti.

Per le configurazioni con più seed c'è anche l'**ensemble** (`..._ensembleN`): la media delle probabilità dei seed
per ogni articolo, decisa prima di vedere i risultati.

Gira da solo alla fine di ogni esperimento della coda: si può guardare anche mentre la coda lavora.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from comune import CLASSI, RESULTS, VERSIONI, metriche_di, per_testata, per_testata_media

SCHEMA = "fold_gruppo"
B = 2000

In [2]:
modelli = {}
for f in sorted(RESULTS.glob(f"feature*_{SCHEMA}_articoli.csv")):
    modelli[f.name.replace(f"_{SCHEMA}_articoli.csv", "")] = pd.read_csv(f)
for d in sorted((RESULTS / "transformer").glob("*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
# ensemble dei seed: per ogni configurazione con più seed finiti, la media delle probabilità per articolo
per_config = {}
for nome in list(modelli):
    m = re.match(r"(.+)_s\d+$", nome)
    if m:
        per_config.setdefault(m[1], []).append(nome)
for config, nomi_seed in per_config.items():
    if len(nomi_seed) > 1:
        parti = [modelli[n].set_index("article_id") for n in nomi_seed]
        e = parti[0][["domain", "fold", "y_true"]].copy()
        e[CLASSI] = sum(p[CLASSI] for p in parti) / len(parti)
        e["y_pred"] = e[CLASSI].to_numpy().argmax(1)
        modelli[f"{config}_ensemble{len(nomi_seed)}"] = e.reset_index()
print("modelli finiti:", list(modelli))
gruppo = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain").gruppo

modelli finiti: ['feature', 'feature_tuned', 'bge_large_v15', 'e5_large_v2', 'modernbert_base_1024_ep2_s1', 'modernbert_base_1024_ep2_s2', 'modernbert_base_1024_ep2_s42', 'modernbert_large_1024_ep2_s1', 'modernbert_large_1024_ep2_s2', 'modernbert_large_1024_ep2_s42', 'modernbert_base_1024_ep2_ensemble3', 'modernbert_large_1024_ep2_ensemble3']


In [3]:
PRINCIPALI = ["f1_macro", "bin_accuratezza", "accuratezza", "tolerant_0.5", "tolerant_1", "bin_f1_macro"]
righe, testate = {}, {}
for nome, p in modelli.items():
    for agg, fun in [("voto", per_testata), ("media", per_testata_media)]:
        t = fun(p); testate[(nome, agg)] = t
        righe[(nome, agg)] = {k: v for k, v in metriche_di(t.y_true, t.y_pred).items() if not isinstance(v, list)}
for f in sorted(RESULTS.glob(f"baseline_*_{SCHEMA}_metriche.json")):
    righe[(f.name.split(f"_{SCHEMA}")[0], "-")] = {k: v for k, v in json.load(open(f))["testata_voto"].items() if not isinstance(v, list)}
tutto = pd.DataFrame(righe).T
tutto.index.names = ["modello", "aggregazione"]
tutto.to_csv(RESULTS / "03_confronto_tutte_le_metriche.csv")
tutto[PRINCIPALI].round(3)

f1_macro  bin_accuratezza  \
modello                             aggregazione                              
feature                             voto             0.377            0.770   
                                    media            0.368            0.776   
feature_tuned                       voto             0.380            0.740   
                                    media            0.367            0.760   
bge_large_v15                       voto             0.325            0.730   
                                    media            0.325            0.730   
e5_large_v2                         voto             0.331            0.727   
                                    media            0.331            0.727   
modernbert_base_1024_ep2_s1         voto             0.334            0.697   
                                    media            0.348            0.730   
modernbert_base_1024_ep2_s2         voto             0.339            0.734   
                                    media            0.335            0.717   
modernbert_base_1024_ep2_s42        voto             0.361            0.717   
                                    media            0.383            0.720   
modernbert_large_1024_ep2_s1        voto             0.326            0.724   
                                    media            0.314            0.714   
modernbert_large_1024_ep2_s2        voto             0.341            0.724   
                                    media            0.350            0.730   
modernbert_large_1024_ep2_s42       voto             0.335            0.701   
                                    media            0.339            0.704   
modernbert_base_1024_ep2_ensemble3  voto             0.371            0.720   
                                    media            0.366            0.717   
modernbert_large_1024_ep2_ensemble3 voto             0.314            0.704   
                                    media            0.348            0.743   
baseline_a_caso                     -                0.214            0.510   
baseline_maggioranza                -                0.096            0.553   

                                                  accuratezza  tolerant_0.5  \
modello                             aggregazione                              
feature                             voto                0.457         0.625   
                                    media               0.454         0.628   
feature_tuned                       voto                0.424         0.597   
                                    media               0.421         0.607   
bge_large_v15                       voto                0.382         0.586   
                                    media               0.382         0.586   
e5_large_v2                         voto                0.391         0.589   
                                    media               0.391         0.589   
modernbert_base_1024_ep2_s1         voto                0.372         0.559   
                                    media               0.378         0.582   
modernbert_base_1024_ep2_s2         voto                0.372         0.567   
                                    media               0.365         0.562   
modernbert_base_1024_ep2_s42        voto                0.414         0.592   
                                    media               0.424         0.599   
modernbert_large_1024_ep2_s1        voto                0.385         0.577   
                                    media               0.375         0.567   
modernbert_large_1024_ep2_s2        voto                0.395         0.581   
                                    media               0.401         0.589   
modernbert_large_1024_ep2_s42       voto                0.388         0.572   
                                    media               0.391         0.579   
modernbert_base_1024_ep2_ensemble3  voto                0.405         0.586   
                    

## Seed: media e deviazione standard

Gli esperimenti con lo stesso nome a parte il seed (`..._s42`, `..._s1`, `..._s2`).

In [4]:
x = tutto[PRINCIPALI].reset_index()
x["config"] = x.modello.str.replace(r"_s\d+$", "", regex=True)
seed = x.groupby(["config", "aggregazione"])[PRINCIPALI].agg(["mean", "std", "count"])
seed.to_csv(RESULTS / "03_confronto_seed.csv")
seed.round(3)

f1_macro               \
                                                     mean    std count   
config                              aggregazione                         
baseline_a_caso                     -               0.214    NaN     1   
baseline_maggioranza                -               0.096    NaN     1   
bge_large_v15                       media           0.325    NaN     1   
                                    voto            0.325    NaN     1   
e5_large_v2                         media           0.331    NaN     1   
                                    voto            0.331    NaN     1   
feature                             media           0.368    NaN     1   
                                    voto            0.377    NaN     1   
feature_tuned                       media           0.367    NaN     1   
                                    voto            0.380    NaN     1   
modernbert_base_1024_ep2            media           0.355  0.025     3   
                                    voto            0.345  0.015     3   
modernbert_base_1024_ep2_ensemble3  media           0.366    NaN     1   
                                    voto            0.371    NaN     1   
modernbert_large_1024_ep2           media           0.334  0.018     3   
                                    voto            0.334  0.007     3   
modernbert_large_1024_ep2_ensemble3 media           0.348    NaN     1   
                                    voto            0.314    NaN     1   

                                                 bin_accuratezza               \
                                                            mean    std count   
config                              aggregazione                                
baseline_a_caso                     -                      0.510    NaN     1   
baseline_maggioranza                -                      0.553    NaN     1   
bge_large_v15                       media                  0.730    NaN     1   
                                    voto                   0.730    NaN     1   
e5_large_v2                         media                  0.727    NaN     1   
                                    voto                   0.727    NaN     1   
feature                             media                  0.776    NaN     1   
                                    voto                   0.770    NaN     1   
feature_tuned                       media                  0.760    NaN     1   
                                    voto                   0.740    NaN     1   
modernbert_base_1024_ep2            media                  0.723  0.007     3   
                                    voto                   0.716  0.018     3   
modernbert_base_1024_ep2_ensemble3  media                  0.717    NaN     1   
                                    voto                   0.720    NaN     1   
modernbert_large_1024_ep2           media                  0.716  0.013     3   
                                    voto                   0.716  0.013     3   
modernbert_large_1024_ep2_ensemble3 media                  0.743    NaN     1   
                                    voto                   0.704    NaN     1   

                                                 accuratezza               \
                                                        mean    std count   
config                              aggregazione                            
baseline_a_caso                     -                  0.243    NaN     1   
baseline_maggioranza                -                  0.316    NaN     1   
bge_large_v15                       media              0.382    NaN     1   
                                    voto               0.382    NaN     1   
e5_large_v2                         media              0.391    NaN     1   
                                    voto               0.391    NaN     1   
feature                             media              0.454    NaN     1   
                      

## Intervalli e differenze con il feature (bootstrap per gruppo)

In [5]:
rif = ("feature_tuned", "voto") if ("feature_tuned", "voto") in testate else ("feature", "voto")   # il feature al suo meglio, se c'è
nomi = [k for k in testate if k[1] == "media" or k == rif]
dom = testate[rif].index
g = gruppo.reindex(dom).to_numpy()
gruppi = np.unique(g)
pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
campioni = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def due(t, idx):
    m = metriche_di(t.y_true.to_numpy()[idx], t.y_pred.to_numpy()[idx])
    return m["f1_macro"], m["bin_accuratezza"]


boot = {k: np.array([due(testate[k].reindex(dom), idx) for idx in campioni]) for k in nomi}
righe = {}
for k in nomi:
    b = boot[k]
    r = {"F1 macro": tutto.loc[k, "f1_macro"], "F1 2.5%": np.percentile(b[:, 0], 2.5), "F1 97.5%": np.percentile(b[:, 0], 97.5),
         "binario": tutto.loc[k, "bin_accuratezza"], "bin 2.5%": np.percentile(b[:, 1], 2.5), "bin 97.5%": np.percentile(b[:, 1], 97.5)}
    if k != rif:
        d = b - boot[rif]
        r.update({"ΔF1 vs feature": tutto.loc[k, "f1_macro"] - tutto.loc[rif, "f1_macro"], "ΔF1 2.5%": np.percentile(d[:, 0], 2.5),
                  "ΔF1 97.5%": np.percentile(d[:, 0], 97.5), "P(ΔF1>0)": (d[:, 0] > 0).mean(),
                  "Δbin vs feature": tutto.loc[k, "bin_accuratezza"] - tutto.loc[rif, "bin_accuratezza"],
                  "Δbin 2.5%": np.percentile(d[:, 1], 2.5), "Δbin 97.5%": np.percentile(d[:, 1], 97.5), "P(Δbin>0)": (d[:, 1] > 0).mean()})
    righe[f"{k[0]} ({k[1]})"] = r
intervalli = pd.DataFrame(righe).T
intervalli.to_csv(RESULTS / "03_confronto_intervalli.csv")
intervalli.round(3)

,F1 macro,F1 2.5%,F1 97.5%,binario,bin 2.5%,bin 97.5%,ΔF1 vs feature,ΔF1 2.5%,ΔF1 97.5%,P(ΔF1>0),Δbin vs feature,Δbin 2.5%,Δbin 97.5%,P(Δbin>0)
feature (media),0.368,0.307,0.429,0.776,0.714,0.833,-0.013,-0.062,0.032,0.299,0.036,0.004,0.068,0.984
feature_tuned (voto),0.380,0.318,0.439,0.740,0.676,0.801,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_tuned (media),0.367,0.306,0.424,0.760,0.696,0.818,-0.013,-0.046,0.016,0.202,0.020,-0.003,0.045,0.944
bge_large_v15 (media),0.325,0.260,0.384,0.730,0.667,0.790,-0.056,-0.112,0.001,0.028,-0.010,-0.063,0.044,0.346
e5_large_v2 (media),0.331,0.268,0.389,0.727,0.661,0.795,-0.049,-0.111,0.011,0.052,-0.013,-0.074,0.043,0.317
modernbert_base_1024_ep2_s1 (media),0.348,0.287,0.407,0.730,0.669,0.791,-0.033,-0.090,0.020,0.120,-0.010,-0.057,0.035,0.315
modernbert_base_1024_ep2_s2 (media),0.335,0.272,0.392,0.717,0.653,0.779,-0.046,-0.104,0.010,0.055,-0.023,-0.070,0.024,0.164
modernbert_base_1024_ep2_s42 (media),0.383,0.319,0.444,0.720,0.656,0.782,0.003,-0.057,0.065,0.551,-0.020,-0.068,0.029,0.190
modernbert_large_1024_ep2_s1 (media),0.314,0.255,0.372,0.714,0.646,0.785,-0.066,-0.135,-0.003,0.021,-0.026,-0.086,0.032,0.162
modernbert_large_1024_ep2_s2 (media),0.350,0.292,0.405,0.730,0.667,0.793,-0.031,-0.096,0.038,0.203,-0.010,-0.072,0.051,0.370


## Per livello e matrici di confusione (testata, media delle probabilità; feature col voto)

In [6]:
LIV = ["l1", "l2", "l3", "l4", "l5"]
per_livello = pd.DataFrame({f"{k[0]} ({k[1]})": {f"{l} {q}": tutto.loc[k, f"{l}_{q}"] for l in LIV for q in ["recall", "f1"]} for k in nomi}).T
display(per_livello.round(2))
for k in nomi:
    t = testate[k]
    print(f"\n{k[0]} ({k[1]})")
    print(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]).to_string())

,l1 recall,l1 f1,l2 recall,l2 f1,l3 recall,l3 f1,l4 recall,l4 f1,l5 recall,l5 f1
feature (media),0.70,0.63,0.37,0.37,0.06,0.08,0.57,0.52,0.19,0.23
feature_tuned (voto),0.65,0.63,0.49,0.44,0.14,0.16,0.38,0.40,0.28,0.28
feature_tuned (media),0.66,0.64,0.46,0.43,0.08,0.09,0.41,0.42,0.25,0.26
bge_large_v15 (media),0.62,0.59,0.37,0.37,0.17,0.17,0.41,0.41,0.08,0.09
e5_large_v2 (media),0.59,0.58,0.35,0.37,0.06,0.06,0.47,0.46,0.19,0.19
modernbert_base_1024_ep2_s1 (media),0.61,0.57,0.42,0.40,0.14,0.13,0.28,0.34,0.36,0.30
modernbert_base_1024_ep2_s2 (media),0.55,0.51,0.40,0.39,0.08,0.08,0.28,0.35,0.44,0.34
modernbert_base_1024_ep2_s42 (media),0.54,0.55,0.57,0.48,0.17,0.16,0.40,0.45,0.28,0.28
modernbert_large_1024_ep2_s1 (media),0.65,0.58,0.25,0.27,0.03,0.04,0.42,0.41,0.31,0.28
modernbert_large_1024_ep2_s2 (media),0.63,0.56,0.26,0.29,0.08,0.10,0.46,0.45,0.36,0.35



feature (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1        50  11   2   8   0
l2        25  24   1  13   2
l3         7   8   2  18   1
l4         3  17   7  55  14
l5         2   5   0  22   7

feature_tuned (voto)
predetto  l1  l2  l3  l4  l5
vero                        
l1        46  13   3   8   1
l2        16  32   4  11   2
l3         7   8   5  15   1
l4         4  21  14  36  21
l5         3   7   2  14  10

feature_tuned (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1        47  13   3   8   0
l2        17  30   4  11   3
l3         7   9   3  16   1
l4         3  17  16  39  21
l5         2   6   2  17   9

bge_large_v15 (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1        44  13   3   9   2
l2        18  24   9   9   5
l3         4  10   6  16   0
l4         8  13  17  39  19
l5         4   6   1  22   3

e5_large_v2 (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1        42  12   5   8